# Modulo 9 · Lezione 6 — API REST con FastAPI

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | Anatomia di una API REST |
| 2 | FastAPI: app, rotte, Pydantic |
| 3 | Endpoint `POST /predict` |
| 4 | Avviare il server con uvicorn |
| 5 | Testare l'API con `requests` |

In [ ]:
# Installazione (se non presenti)
# !pip install fastapi uvicorn[standard] pydantic requests

print("Librerie necessarie:")
import subprocess, sys
paquetes = ['fastapi', 'uvicorn', 'pydantic', 'requests']
for p in paquetes:
    resultado = subprocess.run([sys.executable, '-c', f'import {p}; print({p}.__version__)'],
                               capture_output=True, text=True)
    version = resultado.stdout.strip() if resultado.returncode == 0 else 'NON INSTALLATO'
    print(f"  {p:15}: {version}")

---
## 1. Anatomia di una API REST

```
Cliente                     Server (FastAPI)
-------                     ------------------
POST /predict               @app.post("/predict")
Content-Type: application/json
Body: {"alcohol": 12.5, ...}   → Pydantic valida
                               → modelo.predict()
                            ← 200 OK
                            ← {"prediccion": "bueno", ...}
```

**Verbi HTTP più usati nelle API di ML:**

| Verbo | Uso |
|-------|-----|
| `GET`    | Leggere informazioni (health check, metadati del modello) |
| `POST`   | Inviare dati per ottenere una predizione |
| `DELETE` | Eliminare un record (predizione salvata) |

**Codici di stato HTTP chiave:**
- `200 OK` — tutto ok
- `422 Unprocessable Entity` — dati di input non validi (Pydantic)
- `500 Internal Server Error` — errore nel server

---
## 2. Creare l'applicazione FastAPI

In [ ]:
%%writefile api/main.py
"""
API REST per la predizione della qualità dei vini.
Modelli addestrati in M9_C4 (ML) e M9_C5 (DL).
"""
from pathlib import Path
import json
import numpy as np
import joblib
from datetime import datetime
from typing import Optional, Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, model_validator

app = FastAPI(
    title="Wine Quality API",
    description="Predice se un vino è di buona qualità e il suo punteggio stimato.",
    version="1.0.0"
)

# ---------- Modelli Pydantic ----------

class VinoEntrada(BaseModel):
    """Caratteristiche fisico-chimiche di un vino."""
    tipo:                  Literal['tinto', 'blanco']
    fixed_acidity:         float = Field(..., gt=0, le=20,  description="Acidità fissa (g/L)")
    volatile_acidity:      float = Field(..., ge=0, le=2,   description="Acidità volatile (g/L)")
    citric_acid:           float = Field(..., ge=0, le=1,   description="Acido citrico (g/L)")
    residual_sugar:        float = Field(..., ge=0, le=65,  description="Zucchero residuo (g/L)")
    chlorides:             float = Field(..., ge=0, le=1,   description="Cloruri (g/L)")
    free_sulfur_dioxide:   float = Field(..., ge=0, le=300, description="SO2 libera (mg/L)")
    total_sulfur_dioxide:  float = Field(..., ge=0, le=500, description="SO2 totale (mg/L)")
    density:               float = Field(..., ge=0.98, le=1.05, description="Densità (g/cm³)")
    pH:                    float = Field(..., ge=2.5, le=4.5,   description="pH")
    sulphates:             float = Field(..., ge=0, le=2,        description="Solfati (g/L)")
    alcohol:               float = Field(..., ge=8, le=15,       description="Alcol (%vol)")

    @model_validator(mode='after')
    def validar_so2(self):
        if self.free_sulfur_dioxide > self.total_sulfur_dioxide:
            raise ValueError("free_sulfur_dioxide non può superare total_sulfur_dioxide")
        return self

    class Config:
        json_schema_extra = {
            "example": {
                "tipo": "tinto", "fixed_acidity": 7.4, "volatile_acidity": 0.7,
                "citric_acid": 0.0, "residual_sugar": 1.9, "chlorides": 0.076,
                "free_sulfur_dioxide": 11.0, "total_sulfur_dioxide": 34.0,
                "density": 0.9978, "pH": 3.51, "sulphates": 0.56, "alcohol": 9.4
            }
        }

class PrediccionSalida(BaseModel):
    prediccion:          str
    probabilidad:        float
    puntuacion_estimada: Optional[float] = None
    modelo:              str
    timestamp:           str

# ---------- Caricamento modelli (lazy) ----------

_modelos = {}

FEATURES_ORDEN = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'so2_ratio', 'acid_index', 'density_diff', 'tipo_encoded'
]

def cargar_modelo_rf():
    if 'rf' not in _modelos:
        ruta = Path('models/rf_vinos_v1.joblib')
        if not ruta.exists():
            raise HTTPException(500, "Modello RF non trovato. Esegui prima M9_C4.")
        _modelos['rf'] = joblib.load(ruta)
    return _modelos['rf']

def entrada_a_array(vino: VinoEntrada) -> np.ndarray:
    """Converte il modello Pydantic in un array con le feature derivate."""
    d = {
        'fixed acidity':        vino.fixed_acidity,
        'volatile acidity':     vino.volatile_acidity,
        'citric acid':          vino.citric_acid,
        'residual sugar':       vino.residual_sugar,
        'chlorides':            vino.chlorides,
        'free sulfur dioxide':  vino.free_sulfur_dioxide,
        'total sulfur dioxide': vino.total_sulfur_dioxide,
        'density':              vino.density,
        'pH':                   vino.pH,
        'sulphates':            vino.sulphates,
        'alcohol':              vino.alcohol,
    }
    d['so2_ratio']    = d['free sulfur dioxide'] / (d['total sulfur dioxide'] + 1e-8)
    d['acid_index']   = d['fixed acidity'] - d['volatile acidity']
    d['density_diff'] = abs(d['density'] - 1.0)
    d['tipo_encoded'] = 1 if vino.tipo == 'tinto' else 0
    return np.array([[d[f] for f in FEATURES_ORDEN]])

# ---------- Rotte ----------

@app.get("/")
def raiz():
    return {"mensaje": "Wine Quality API v1.0", "docs": "/docs"}

@app.get("/health")
def health():
    return {"status": "ok", "timestamp": datetime.now().isoformat()}

@app.get("/model/info")
def info_modelo():
    meta_path = Path('models/rf_vinos_v1_meta.json')
    if not meta_path.exists():
        raise HTTPException(404, "Metadati non trovati")
    with open(meta_path, encoding='utf-8') as f:
        return json.load(f)

@app.post("/predict", response_model=PrediccionSalida)
def predecir(vino: VinoEntrada):
    """Predice se un vino è di buona qualità (quality >= 7)."""
    modelo = cargar_modelo_rf()
    X = entrada_a_array(vino)

    pred = int(modelo.predict(X)[0])
    prob = float(modelo.predict_proba(X)[0, 1])

    return PrediccionSalida(
        prediccion          = 'bueno' if pred == 1 else 'normal',
        probabilidad        = round(prob, 4),
        puntuacion_estimada = None,  # aggiunto nell'esercizio con il modello DL
        modelo              = 'rf_v1',
        timestamp           = datetime.now().isoformat()
    )

@app.post("/predict/batch")
def predecir_batch(vinos: list[VinoEntrada]):
    """Predice per una lista di vini (massimo 100 alla volta)."""
    if len(vinos) > 100:
        raise HTTPException(400, "Massimo 100 vini per richiesta")
    modelo = cargar_modelo_rf()
    X = np.vstack([entrada_a_array(v) for v in vinos])
    preds = modelo.predict(X).tolist()
    probs = modelo.predict_proba(X)[:, 1].tolist()
    return [
        {"prediccion": 'bueno' if p == 1 else 'normal',
         "probabilidad": round(pr, 4)}
        for p, pr in zip(preds, probs)
    ]

---
## 3. Avviare il server

In [ ]:
import subprocess, time, signal, os

# Avviare uvicorn in background dalla directory radice del progetto
proceso = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--port', '8000', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)  # attendere l'avvio

if proceso.poll() is None:
    print(f"✓ Server avviato (PID={proceso.pid})")
    print("  URL: http://localhost:8000")
    print("  Docs: http://localhost:8000/docs")
else:
    stderr = proceso.stderr.read().decode()
    print(f"✗ Errore all'avvio:\n{stderr}")

---
## 4. Testare l'API con `requests`

In [ ]:
import requests

BASE = 'http://localhost:8000'

# GET /health
r = requests.get(f'{BASE}/health')
print(f"GET /health  →  {r.status_code}")
print(f"  {r.json()}")

# GET /model/info
r = requests.get(f'{BASE}/model/info')
print(f"\nGET /model/info  →  {r.status_code}")
info = r.json()
if 'metricas_test' in info:
    print(f"  Metriche: {info['metricas_test']}")

In [ ]:
# POST /predict — vino di esempio
vino_ejemplo = {
    "tipo": "tinto",
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "pH": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}

r = requests.post(f'{BASE}/predict', json=vino_ejemplo)
print(f"POST /predict  →  {r.status_code}")
pred = r.json()
for k, v in pred.items():
    print(f"  {k}: {v}")

In [ ]:
# Testare la validazione di Pydantic — dati errati
vino_invalido = {
    **vino_ejemplo,
    "alcohol": 99,           # fuori range (>15)
    "free_sulfur_dioxide": 200,  # maggiore del totale (100)
    "total_sulfur_dioxide": 100
}

r = requests.post(f'{BASE}/predict', json=vino_invalido)
print(f"POST con dati non validi  →  {r.status_code}")
print(f"  Errore: {r.json()['detail'][0]['msg']}")

In [ ]:
# POST /predict/batch — più vini
import pandas as pd

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
muestra = df.head(5)

vinos_batch = []
for _, row in muestra.iterrows():
    vinos_batch.append({
        "tipo":                  row['tipo'],
        "fixed_acidity":         row['fixed acidity'],
        "volatile_acidity":      row['volatile acidity'],
        "citric_acid":           row['citric acid'],
        "residual_sugar":        row['residual sugar'],
        "chlorides":             row['chlorides'],
        "free_sulfur_dioxide":   row['free sulfur dioxide'],
        "total_sulfur_dioxide":  row['total sulfur dioxide'],
        "density":               row['density'],
        "pH":                    row['pH'],
        "sulphates":             row['sulphates'],
        "alcohol":               row['alcohol'],
    })

r = requests.post(f'{BASE}/predict/batch', json=vinos_batch)
print(f"POST /predict/batch ({len(vinos_batch)} vini)  →  {r.status_code}")
for i, resultado in enumerate(r.json()):
    real = int(muestra.iloc[i]['buen_vino'])
    print(f"  Vino {i+1}: pred={resultado['prediccion']:6}  prob={resultado['probabilidad']:.3f}  real={'bueno' if real else 'normal'}")

In [ ]:
# Fermare il server
proceso.terminate()
proceso.wait()
print("✓ Server fermato")

---
## Pratica

### Esercizio 1 — Endpoint `/predict/dl` con il modello TensorFlow

**Consegna:** Aggiungi a `api/main.py` un nuovo endpoint `POST /predict/dl` che:

1. Carichi `models/dl_clf_v1.keras` e `models/dl_scaler.joblib`
2. Restituisca la predizione con `"modelo": "dl_v1"`
3. Includa inoltre il campo `"puntuacion_estimada"` usando `models/dl_reg_v1.keras`
4. Testa entrambi gli endpoint e confronta i risultati

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

Aggiungi alla fine di `api/main.py`:

```python
from tensorflow import keras as tf_keras

def cargar_modelos_dl():
    if 'dl_clf' not in _modelos:
        _modelos['dl_clf'] = tf_keras.models.load_model('models/dl_clf_v1.keras')
        _modelos['dl_reg'] = tf_keras.models.load_model('models/dl_reg_v1.keras')
        _modelos['dl_sc']  = joblib.load('models/dl_scaler.joblib')

@app.post("/predict/dl", response_model=PrediccionSalida)
def predecir_dl(vino: VinoEntrada):
    """Predizione usando la rete neurale (TensorFlow/Keras)."""
    cargar_modelos_dl()
    X_raw  = entrada_a_array(vino)                            # non scalato
    X_sc   = _modelos['dl_sc'].transform(X_raw).astype('float32')
    prob   = float(_modelos['dl_clf'].predict(X_sc, verbose=0)[0, 0])
    punt   = float(_modelos['dl_reg'].predict(X_sc, verbose=0)[0, 0])
    return PrediccionSalida(
        prediccion          = 'bueno' if prob >= 0.5 else 'normal',
        probabilidad        = round(prob, 4),
        puntuacion_estimada = round(punt, 2),
        modelo              = 'dl_v1',
        timestamp           = datetime.now().isoformat()
    )
```

Poi testa dal notebook:
```python
r_dl = requests.post('http://localhost:8000/predict/dl', json=vino_ejemplo)
print(r_dl.json())
```
</details>

---
### Esercizio 2 — Middleware di logging

**Consegna:** FastAPI ha un sistema di middleware che intercetta tutte le richieste. Aggiungine uno che registri in `logs/api_requests.log` ogni richiesta con:
- timestamp
- metodo (GET/POST)
- rotta (`/predict`)
- codice di risposta
- tempo di risposta in ms

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

Aggiungi a `api/main.py` dopo `app = FastAPI(...)`:

```python
import time, logging
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request

logging.basicConfig(
    filename='logs/api_requests.log',
    level=logging.INFO,
    format='%(message)s'
)

class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        inicio = time.perf_counter()
        respuesta = await call_next(request)
        duracion_ms = (time.perf_counter() - inicio) * 1000
        logging.info(
            f"{datetime.now().isoformat()} | {request.method} {request.url.path} "
            f"| {respuesta.status_code} | {duracion_ms:.1f}ms"
        )
        return respuesta

app.add_middleware(LoggingMiddleware)
```
</details>

---
### Esercizio 3 — Test di carico

**Consegna:** Simula 100 richieste all'endpoint `/predict` e analizza le prestazioni:

1. Invia 100 richieste una a una misurando il tempo di ciascuna
2. Calcola: media, mediana, percentile 95 e massimo del tempo di risposta
3. Quante richieste al secondo regge l'API?
4. Usa `concurrent.futures.ThreadPoolExecutor` per inviarne 10 in parallelo e confronta

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import requests, time, numpy as np
from concurrent.futures import ThreadPoolExecutor

URL = 'http://localhost:8000/predict'

def enviar_peticion(_):
    t0 = time.perf_counter()
    r = requests.post(URL, json=vino_ejemplo)
    return (time.perf_counter() - t0) * 1000, r.status_code

# Sequenziale
print("Sequenziale (100 richieste):")
t0 = time.perf_counter()
tiempos = [enviar_peticion(None)[0] for _ in range(100)]
total = time.perf_counter() - t0
tiempos = np.array(tiempos)
print(f"  Tempo totale: {total:.1f}s  ({100/total:.1f} req/s)")
print(f"  Media: {tiempos.mean():.1f}ms  Mediana: {np.median(tiempos):.1f}ms  P95: {np.percentile(tiempos,95):.1f}ms  Max: {tiempos.max():.1f}ms")

# Parallelo
print("\nParallelo (10 worker, 100 richieste):")
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as pool:
    resultados = list(pool.map(enviar_peticion, range(100)))
total_p = time.perf_counter() - t0
tiempos_p = np.array([r[0] for r in resultados])
print(f"  Tempo totale: {total_p:.1f}s  ({100/total_p:.1f} req/s)")
print(f"  Media: {tiempos_p.mean():.1f}ms  P95: {np.percentile(tiempos_p,95):.1f}ms")
```
</details>

---
## Riepilogo

```python
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Mi API")

class Entrada(BaseModel):
    valor: float = Field(..., ge=0, le=100, description="Valore tra 0 e 100")

class Salida(BaseModel):
    resultado: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=Salida)
def predecir(datos: Entrada):
    return Salida(resultado="positivo", score=datos.valor / 100)

# Avviare: uvicorn api.main:app --reload --port 8000
```

## Cosa vedremo nella prossima lezione?
Nella **Lezione 7** documenteremo il progetto: docstring, README, OpenAPI/Swagger e flusso base di Git/GitHub.